In [30]:
from pathlib import Path
import subprocess

def project_root() -> Path:
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"],
            stderr=subprocess.STDOUT,
            text=True,
        ).strip()
        return Path(out).resolve()
    except Exception:
        return Path.cwd().resolve()

ROOT = project_root()
ROOT


WindowsPath('C:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai')

In [31]:
CSV_PATH = ROOT / "data" / "raw" / "fakenewsnet" / "dataset" / "politifact_fake.csv"
CSV_PATH.exists()
CSV_PATH


WindowsPath('C:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/raw/fakenewsnet/dataset/politifact_fake.csv')

In [34]:
import pandas as pd
import time
import sys
import threading
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import time

NUM_THREAD = 14  # keep < 13

sys.path.insert(0, "../")

import article_scraper
importlib.reload(article_scraper)

from article_scraper import scrap_from_web, _requests_session_with_retries

print("Loaded from:", article_scraper.__file__)

df = pd.read_csv(
    CSV_PATH,
    usecols=["id", "news_url", "title"],
    dtype=str,
    keep_default_na=False,
)

df50 = df.head(50).copy()

_tls = threading.local()

def _get_thread_session():
    if getattr(_tls, "session", None) is None:
        _tls.session = _requests_session_with_retries(total_retries=2, backoff_factor=0.5)
    return _tls.session

def _worker(i: int, url: str):
    t0 = time.time()
    try:
        sess = _get_thread_session()
        # txt, meta = scrap_from_web(
        #     url,
        #     timeout=10.0,
        #     max_attempts=2,
        #     use_wayback=True,
        #     session=sess,
        #     respect_robots=True,
        #     return_meta=True,
        # )
        txt, meta = scrap_from_web(
            url,
            timeout=(3.0, 8.0),
            max_attempts=1,
            use_wayback=False,
            session=sess,
            respect_robots=True,
            return_meta=True,
        )
        dt = time.time() - t0

        fetch_status = meta.get("fetch_status")
        http_status = meta.get("http_status")
        used_wayback = bool(meta.get("used_wayback", False))

        ok = bool(txt and txt.strip())
        status_str = (
            f"{'ok' if ok else 'fail'} | {fetch_status} | http={http_status} | "
            f"wayback={used_wayback} ({dt:.1f}s)"
        )
        return i, txt, status_str, fetch_status, http_status, used_wayback

    except Exception as e:
        dt = time.time() - t0
        return i, None, f"error:{type(e).__name__}: {e} ({dt:.1f}s)", "worker_exception", None, False

urls = df50["news_url"].tolist()

raw_texts = [None] * len(urls)
status_strs = [None] * len(urls)
fetch_statuses = [None] * len(urls)
http_statuses = [None] * len(urls)
used_waybacks = [None] * len(urls)

with ThreadPoolExecutor(max_workers=NUM_THREAD) as ex:
    futures = [ex.submit(_worker, i, u) for i, u in enumerate(urls)]
    for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Scraping x{NUM_THREAD}"):
        i, txt, status_str, fstat, hstat, wb = fut.result()
        raw_texts[i] = txt
        status_strs[i] = status_str
        fetch_statuses[i] = fstat
        http_statuses[i] = hstat
        used_waybacks[i] = wb

df50["raw_text"] = raw_texts
df50["scrape_status"] = status_strs
df50["fetch_status"] = fetch_statuses
df50["http_status"] = http_statuses
df50["used_wayback"] = used_waybacks

print(df50["fetch_status"].value_counts(dropna=False).head(25))
print(df50["http_status"].value_counts(dropna=False).head(25))

df50[["id", "news_url", "scrape_status", "fetch_status", "http_status", "used_wayback"]].head(10)


Loaded from: c:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\unified_schema\testings\..\article_scraper.py


Scraping x14: 100%|██████████| 50/50 [00:30<00:00,  1.65it/s]

fetch_status
success           23
http_404          10
http_403           6
error              5
200_no_parse       2
robots_blocked     2
http_503           1
timeout            1
Name: count, dtype: int64
http_status
200.0    25
404.0    10
NaN       8
403.0     6
503.0     1
Name: count, dtype: int64


,id,news_url,scrape_status,fetch_status,http_status,used_wayback
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,fail | http_404 | http=404 | wayback=False (2.8s),http_404,404.0,False
1,politifact15156,politics2020.info/index.php/2018/03/13/court-o...,fail | error | http=None | wayback=False (2.0s),error,NaN,False
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467...,ok | success | http=200 | wayback=False (1.5s),success,200.0,False
3,politifact14355,https://howafrica.com/oscar-pistorius-attempts...,fail | http_404 | http=404 | wayback=False (0.7s),http_404,404.0,False
4,politifact15371,http://washingtonsources.org/trump-votes-for-d...,ok | success | http=200 | wayback=False (1.7s),success,200.0,False
5,politifact14404,gloria.tv/video/yRrtUtTCfPga6cq2VDJPcgQe4,fail | http_404 | http=404 | wayback=False (1.6s),http_404,404.0,False
6,politifact13919,http://blogs.trendolizer.com/2015/01/new-york-...,ok | success | http=200 | wayback=False (0.5s),success,200.0,False
7,politifact14795,https://web.archive.org/web/20171027105356/htt...,ok | success | http=200 | wayback=False (1.9s),success,200.0,False
8,politifact14328,https://web.archive.org/web/20170702174006/htt...,ok | success | http=200 | wayback=False (2.0s),success,200.0,False
9,politifact13775,http://beforeitsnews.com/opinion-conservative/...,ok | success | http=200 | wayback=False (1.0s),success,200.0,False


In [33]:
import json
from pathlib import Path

OUT_DIR = Path.cwd() / "web_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSONL = OUT_DIR / "politifact_fake_first50_raw.jsonl"
OUT_PARQUET = OUT_DIR / "politifact_fake_first50_raw.parquet"

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for _, r in df50.iterrows():
        f.write(json.dumps({
            "id": r["id"],
            "news_url": r["news_url"],
            "scrape_status": r["scrape_status"],
            "fetch_status": r.get("fetch_status"),
            "http_status": r.get("http_status"),
            "used_wayback": r.get("used_wayback"),
            "raw_text": r["raw_text"],
        }, ensure_ascii=False) + "\n")

# fastparquet avoids your pyarrow extension-type error
df50.to_parquet(OUT_PARQUET, index=False, engine="fastparquet")

OUT_JSONL, OUT_PARQUET


(WindowsPath('c:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/unified_schema/testings/web_result/politifact_fake_first50_raw.jsonl'),
 WindowsPath('c:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/unified_schema/testings/web_result/politifact_fake_first50_raw.parquet'))